# Introduction to Weaviate

Up until now, we've performed much of the RAG workflow manually, particularly the retrieval step. From now on, we'll use Weaviate to embed and store documents for retrieval, and orchestrate the full RAG workflow.

**Run the first few cells to install `weaviate-client` and load the `weaviate/agents` dataset from Hugging Face and subset the first `200` rows.**

In [ ]:
# Install weaviate-client...

## TASK 1
from datasets import load_dataset

movies_dataset = load_dataset("weaviate/agents", "personalization-agent-movies", split="train")

print(f"Dataset: {movies_dataset.config_name}")
for o in movies_dataset[:2]["properties"]:
    print(o)
    
movies_data = movies_dataset[:1000]["properties"]

## TASK 2
import weaviate
from openai import OpenAI as OpenAIClient

raw_client = OpenAIClient()
API_KEY  = str(raw_client.api_key)
API_BASE = str(raw_client.base_url)

HEADERS = {
    "X-OpenAI-Api-Key": API_KEY,
    "X-OpenAI-BaseURL": API_BASE.rstrip("/").removesuffix("/v1"),
}

try:
    client = weaviate.connect_to_embedded(
        version="1.32.3",
        headers=HEADERS,
        environment_variables={"LOG_LEVEL": "error"},
    )
except weaviate.exceptions.WeaviateStartUpError:
    # If already running, connect to it instead
    client = weaviate.connect_to_local(
        port=8079,
        grpc_port=50050,
        headers=HEADERS,
    )

client.is_ready()
client.get_meta()["version"]

## TASK 3
from weaviate.classes.config import Property, DataType, Configure

# Delete "Movies" if it's already present to allow repeated runs
if "Movies" in client.collections.list_all():
    client.collections.delete("Movies")

client.collections.create(
    name="Movies",
    properties=[
        Property(
            name="title",
            data_type=DataType.TEXT,
        ),
        Property(
            name="overview",
            data_type=DataType.TEXT,
        ),
        Property(
            name="release_year",
            data_type=DataType.INT,
        ),
        Property(
            name="popularity",
            data_type=DataType.NUMBER,
        ),
    ],
    vector_config=[
        Configure.Vectors.text2vec_openai(
            name="default",
            source_properties=["title", "overview"],
            model="text-embedding-3-small"
        )
    ]
)

## TASK 4
movies = client.collections.get("Movies")

def preprocess_row(item: dict) -> dict:
    obj = {
        "title": item["title"],
        "overview": item["overview"],
        "release_year": item["release_date"].year,
        "popularity": item["popularity"],
    }
    return obj


for i, item in enumerate(movies_data[:2]):
    print(preprocess_row(item))

from tqdm import tqdm
from weaviate.util import generate_uuid5

with movies.batch.fixed_size(batch_size=100) as batch:
    for item in tqdm(movies_data):
        obj = preprocess_row(item)

        # Add object to batch for import
        batch.add_object(
            properties=obj,
            uuid=generate_uuid5(item["title"]+item["overview"])
        )

if movies.batch.failed_objects:
    print(f"{len(movies.batch.failed_objects)} objects failed during import:")
    for failed in movies.batch.failed_objects[:3]:
        print(failed.message)

print(f"Collection size: {len(movies)}")
client.close()


In [6]:
!pip install weaviate-client==4.16.10 pydantic==2.11.9

### Loading the Dataset

We'll be using a pre-arranged `movies` dataset. Let's load it and take a look:

In [7]:
from datasets import load_dataset

movies_dataset = load_dataset("weaviate/agents", "personalization-agent-movies", split="train")

In [8]:
print(f"Dataset: {movies_dataset.config_name}")
for o in movies_dataset[:2]["properties"]:
    print(o)

To speed things up, we'll get a smaller subset of rows to work with.

In [9]:
movies_data = movies_dataset[:200]["properties"]

### Set up Weaviate

Let's install the Weaviate client library, and set up our database. We'll create our own Weaviate database instance locally, with Embedded Weaviate, rather than a Cloud instance. 

**Create a Weaviate client with the `weaviate` module and configuration provided; check it works with the `.is_ready()` and `.get_meta()` client methods.**

In [7]:
import weaviate
from openai import OpenAI as OpenAIClient

raw_client = OpenAIClient()
API_KEY  = str(raw_client.api_key)
API_BASE = str(raw_client.base_url)

HEADERS = {
    "X-OpenAI-Api-Key": API_KEY,
    "X-OpenAI-BaseURL": API_BASE.rstrip("/").removesuffix("/v1"),
}

try:
    client = ____.connect_to_embedded(
        version="1.32.3",
        headers=HEADERS,
        environment_variables={"LOG_LEVEL": "error"},
    )
except weaviate.exceptions.WeaviateStartUpError:
    # If already running, connect to it instead
    client = weaviate.connect_to_local(
        port=8079,
        grpc_port=50050,
        headers=HEADERS,
    )

Check that the client connection is ready:

In [6]:
client.____()

And the Weaviate version:

In [7]:
client.____()["version"]

### Create a collection

Let's create a simple collection (the NoSQL equivalent of table) to add our movie data to.

**Create a collection called `"Movies"`, add another `TEXT` property called `"title"`, and specify an OpenAI embedding model with the `.text2vec_openai()` method.**

In [8]:
from weaviate.classes.config import Property, DataType, Configure

# Delete "Movies" if it's already present to allow repeated runs
if "Movies" in client.collections.list_all():
    client.collections.delete("Movies")

client.collections.create(
    name="____",
    properties=[
        ____(
            name="____",
            data_type=DataType.____,
        ),
        Property(
            name="overview",
            data_type=DataType.TEXT,
        ),
        Property(
            name="release_year",
            data_type=DataType.INT,
        ),
        Property(
            name="popularity",
            data_type=DataType.NUMBER,
        ),
    ],
    vector_config=[
        Configure.Vectors.____(
            name="default",
            source_properties=["title", "overview"],
            model="text-embedding-3-small"
        )
    ]
)

### Insert data

We will insert our data by:

1. Getting our collection object to interact with
2. Preprocessing the raw Movie data to prepare the object to insert
3. Perform insertion

**Get the collection, and complete the loop to preprocess each row and add it to the collection. Finish by closing the client connection.**

In [10]:
movies = client.collections.____("Movies")

Before inserting the data into the collection, we'll perform some preprocessing to select particular columns. This preprocessing function (`preprocess_row()`) has been written for you, along with the loop to apply it.

In [11]:
def preprocess_row(item: dict) -> dict:
    obj = {
        "title": item["title"],
        "overview": item["overview"],
        "release_year": item["release_date"].year,
        "popularity": item["popularity"],
    }
    return obj


for i, item in enumerate(movies_data[:2]):
    print(preprocess_row(item))

With `movies_data` preprocessed, it's time to batch import it into the `movies` collection:

In [12]:
from tqdm import tqdm
from weaviate.util import generate_uuid5

with movies.batch.fixed_size(batch_size=100) as batch:
    for item in tqdm(movies_data):
        obj = preprocess_row(item)

        # Add object to batch for import
        batch.____(
            properties=obj,
            uuid=generate_uuid5(item["title"]+item["overview"])
        )

As a best practice, you should check whether any inserts failed:

In [13]:
if movies.batch.failed_objects:
    print(f"{len(movies.batch.failed_objects)} objects failed during import:")
    for failed in movies.batch.failed_objects[:3]:
        print(failed.message)

Now, the collection count should reflect the new objects:

In [14]:
print(f"Collection size: {len(movies)}")

### Close the client

Remember to run `client.close()` with the Weaviate client library to close the connection and clean up the resources. 

With Embedded Weaviate, `client.close()` will also shut down Weaviate.

In [15]:
client.____()